In [1]:
!pip install -q transformers peft librosa sinling torch fastapi uvicorn pyngrok python-multipart nest-asyncio "torchao>=0.16.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.1 MB/s eta 0:00:00


In [ ]:
import torch
import librosa
import numpy as np
import json
from transformers import (
    WhisperForConditionalGeneration,
    WhisperProcessor,
    pipeline
)
from peft import PeftModel
from sinling import SinhalaTokenizer, POSTagger

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️ Device: {device}")

# ── Whisper ASR ──
print("Loading Whisper ASR...")
BASE_MODEL = "openai/whisper-medium"
ADAPTER_ID = "SPEAK-ASR/whisper-si-exp-10-medium-all"
whisper_processor = WhisperProcessor.from_pretrained(BASE_MODEL)
whisper_model = WhisperForConditionalGeneration.from_pretrained(BASE_MODEL)
whisper_model = PeftModel.from_pretrained(whisper_model, ADAPTER_ID)
whisper_model = whisper_model.merge_and_unload()
whisper_model = whisper_model.to(device)
whisper_model.eval()
print("✅ Whisper ready!")

# ── Sinling ──
print("Loading Sinling...")
sinhala_tokenizer = SinhalaTokenizer()
pos_tagger = POSTagger()
print("✅ Sinling ready!")

# ── Emotion Detection ──
print("Loading Emotion classifier...")
emo_model_id = "r-f/wav2vec-english-speech-emotion-recognition"
# NOTE: loading this checkpoint directly via Wav2Vec2ForSequenceClassification
# left the classifier/projector head randomly initialized — the old load
# report showed "projector.weight/bias" and "classifier.weight/bias" as
# MISSING, meaning predictions were coming from an untrained head (noise).
# The `pipeline()` helper resolves the correct head/preprocessing for this
# checkpoint automatically, which fixes that.
emotion_classifier = pipeline(
    "audio-classification",
    model=emo_model_id,
    device=0 if device == "cuda" else -1,
)
print("✅ Emotion classifier ready!")

print("\n🎉 All models loaded!")


In [3]:
SINHALA_TO_GLOSS = {
    # ── Core pronouns ──
    "මම": "ME",
    "මට": "ME",
    "මගේ": "ME",
    "ඔබ": "YOU",
    "ඔබට": "YOU",
    "ඔහු": "HE",
    "ඇය": "SHE",
    "අපි": "WE",

    # ── Common verbs ──
    "යනවා": "GO",
    "එනවා": "COME",
    "කනවා": "EAT",
    "බොනවා": "DRINK",
    "කරනවා": "DO",
    "බලනවා": "SEE",
    "දැනගන්නවා": "KNOW",
    "ආදරය": "LOVE",
    "හදනවා": "MAKE",
    "හදන්නෙමු": "MAKE",

    # ── Common nouns ──
    "ගෙදර": "HOME",
    "ගෙදරක්": "HOME",
    "පාසැල": "SCHOOL",
    "වතුර": "WATER",
    "කෑම": "FOOD",
    "අම්මා": "MOTHER",
    "තාත්තා": "FATHER",

    # ── Greetings / expressions ──
    "ආයුබෝවන්": "HELLO",
    "ස්තුතියි": "THANK_YOU",
    "සමාවෙන්න": "SORRY",
    "ඔව්": "YES",
    "නැහැ": "NO",
    "හොඳ": "GOOD",
    "නරක": "BAD",

    # ── Time / place / pronouns ──
    "අද": "TODAY",
    "හෙට": "TOMORROW",
    "ඊයේ": "YESTERDAY",
    "ඔයා": "YOU",
    "ඔයාගේ": "YOUR",
    "ඔයාට": "YOU",
    "නම": "NAME",
    "මොකක්ද": "WHAT",
    "පුළුවන්": "CAN",
    "කොහෙද": "WHERE",
}

print(f"📖 Gloss dictionary: {len(SINHALA_TO_GLOSS)} words")

📖 Gloss dictionary: 42 words


In [ ]:
def transcribe(audio_path):
    """Sinhala speech → text using fine-tuned Whisper"""
    waveform, sr = librosa.load(audio_path, sr=16000, mono=True)
    inputs = whisper_processor(waveform, sampling_rate=16000, return_tensors="pt").to(device)

    forced_decoder_ids = whisper_processor.get_decoder_prompt_ids(
        language="sinhala", task="transcribe"
    )

    with torch.no_grad():
        predicted_ids = whisper_model.generate(
            inputs["input_features"],
            forced_decoder_ids=forced_decoder_ids
        )

    text = whisper_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return text


def tokenize_sinhala(text):
    """Tokenize + POS tag using sinling"""
    tokens = sinhala_tokenizer.tokenize(text)
    pos_tags = pos_tagger.predict([tokens])
    return tokens, pos_tags[0] if pos_tags else []


def detect_emotion(audio_path):
    """Classify emotion from audio via the HF audio-classification pipeline
    (see cell-1 note: this replaces a manual model load that had an
    untrained/randomly-initialized classifier head)."""
    results = emotion_classifier(audio_path, top_k=1)
    top = results[0]
    return {"emotion": top["label"], "confidence": round(top["score"], 3)}


def tokens_to_glosses(tokens):
    """Convert Sinhala tokens to sign language glosses"""
    glosses = []
    unknown = []

    for token in tokens:
        gloss = SINHALA_TO_GLOSS.get(token)
        if gloss:
            glosses.append(gloss)
        else:
            unknown.append(token)

    return glosses, unknown


def full_pipeline(audio_path):
    """Run the complete pipeline: audio → glosses"""
    # Step 1: ASR
    text = transcribe(audio_path)

    # Step 2: Tokenize
    tokens, pos_tags = tokenize_sinhala(text)

    # Step 3: Emotion
    emotion = detect_emotion(audio_path)

    # Step 4: Gloss mapping
    glosses, unknown = tokens_to_glosses(tokens)

    return {
        "transcription": text,
        "tokens": tokens,
        "pos_tags": [{"word": w, "tag": t} for w, t in pos_tags],
        "emotion": emotion,
        "glosses": glosses,
        "unknown_tokens": unknown,
    }


print("✅ Pipeline functions ready!")


In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import uvicorn
import tempfile
import os
import threading
from pyngrok import ngrok

app = FastAPI(title="SSL Pipeline API")

# Allow CORS from any origin (needed for browser → Colab)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def health():
    return {"status": "ok", "glosses_available": len(SINHALA_TO_GLOSS)}

@app.post("/translate")
async def translate(audio: UploadFile = File(...)):
    """
    Accept an audio file, run the full pipeline, return glosses.
    """
    # Use the uploaded file's own extension (e.g. .webm/.m4a) instead of
    # hardcoding .wav — librosa/audioread pick the right decoder from the
    # actual file content, but a mismatched suffix is an easy way to hit a
    # decoder that mishandles it, so keep them consistent.
    ext = os.path.splitext(audio.filename or "")[1] or ".wav"
    with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as tmp:
        content = await audio.read()
        tmp.write(content)
        tmp_path = tmp.name

    try:
        result = full_pipeline(tmp_path)
        return JSONResponse(content=result)
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"error": str(e)}
        )
    finally:
        os.unlink(tmp_path)

@app.get("/glosses")
def list_glosses():
    """Return all available gloss mappings"""
    return {
        "dictionary": SINHALA_TO_GLOSS,
        "total": len(SINHALA_TO_GLOSS)
    }

# ── Start server with ngrok ──

# Kill any existing tunnels
ngrok.kill()

# ⚠️ VERY IMPORTANT: Replace YOUR_NGROK_TOKEN with your actual token from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("3DXcrEbBHWr8SjFhpn5WPxY0LD1_dKh9vyxCwNy3FeMRvgkr")

public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"🌐 PUBLIC API URL: {public_url}")
print(f"{'='*60}")
print(f"\nPaste this URL into ssl_renderer.html!")

# ── Auto-register this URL with the Next.js app (optional) ──
# This POSTs the current tunnel URL to /api/config/ngrok-url so the app
# always knows the live address without manual copy-pasting.
#
# Colab can only reach a *publicly reachable* app — it cannot reach
# "http://localhost:3000" on your own laptop. So:
#   - Developing locally? Leave APP_BASE_URL empty (skip this) and instead
#     paste the URL above into the app's Dashboard -> Settings tab.
#   - App deployed somewhere public (e.g. Vercel)? Set APP_BASE_URL below
#     and this cell will keep it in sync automatically on every restart.
APP_BASE_URL = ""  # e.g. "https://your-app.vercel.app"
NGROK_UPDATE_SECRET = "mWx5qfjet1oa4SEhr2-AvFE2sLMqCSz1"  # must match NGROK_UPDATE_SECRET in .env.local

if APP_BASE_URL:
    import requests
    try:
        requests.post(
            f"{APP_BASE_URL}/api/config/ngrok-url",
            json={"url": public_url.public_url},
            headers={"x-api-key": NGROK_UPDATE_SECRET},
            timeout=10,
        )
        print(f"✅ Registered URL with {APP_BASE_URL}")
    except Exception as e:
        print(f"⚠️ Could not auto-register URL: {e}")

# Run Uvicorn in a background thread bypassing asyncio.run()
def run_server():
    import asyncio
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)

    # Create a new event loop for this thread to avoid Colab/nest_asyncio conflicts
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(server.serve())

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("\n✅ Server is running in the background!")
